# Species migration portfolio assignment

## Step 1: Get occurrence data from the Global Biodiversity Information Facility (GBIF)
For your species migration portfolio assignment, you will need to acquire GBIF data for a different species. I recommend choosing a terrestrial species, as this will work best with the ecoregions data used in the assignment.

## Set up
In the imports cell, we’ve included some packages that you will need. Add imports for additional packages that will help you to:
1. Work with reproducible file paths
2. Work with tabular data

In [37]:
### install package pygbif for getting observation data from gbif
! pip install pygbif

import time ### time processing steps
import zipfile ### read and extract .zip folders
from getpass import getpass ### handles sensitive inputs like passwords and API tokens
from glob import glob ### function for finding file paths
import requests ### make requests to websites and web APIs

### GBIF packages
import pygbif.occurrences as occ ### get GBIF occurrence data
import pygbif.species as species ### deals with species names

### import additional packages you'll need
import os
import pathlib

### programmatically rename files
import shutil

### package for tabular data
import pandas as pd

### package for handling geospatial vector data
import geopandas as gpd

### package for folder/ file management
import os

### package for file paths and nested folders
import pathlib

### Step 1A: Make a GBIF account
You will need to make a free account with [GBIF](https://www.gbif.org/) to access their data. Keep track of your username, password, and the email address you sign up with, as you'll need all of this later!

You will need a (free) [GBIF account](https://www.gbif.org/) to complete this challenge. Then, run the following code to enter your credentials for the rest of your session.

This code is **interactive**, meaning that it will **ask you for a response**! The prompt can sometimes be
hard to see if you are using VSCode – it appears at the **top** of your editor window.

**Tips:**
1. If you need to save credentials across multiple sessions, you can consider loading them in from a file like a `.env`, but make sure to add it to .gitignore so you don’t commit your credentials to your repository!
2. If you accidentally enter your credentials wrong, you can set `reset = True` instead of `reset = False`.

**Warning**
Your email address **must** match the email you used to sign up for GBIF!

In [ ]:
### change this to reset = False after you run this chunk the first time
reset = False

####--------------------------------------####
#### DO NOT MODIFY THE REST OF THIS CODE! ####
####--------------------------------------####
### This code asks for your credentials and saves it for the rest of the session.
### NEVER put your credentials into your code!!!!
### GBIF needs a username, password, and email 

### request and store username
if (not ('GBIF_USER'  in os.environ)) or reset:
    os.environ['GBIF_USER'] = input('GBIF username:')

### securely request and store password
if (not ('GBIF_PWD'  in os.environ)) or reset:
    os.environ['GBIF_PWD'] = getpass('GBIF password:')
    
### request and store account email address
if (not ('GBIF_EMAIL'  in os.environ)) or reset:
    os.environ['GBIF_EMAIL'] = input('GBIF email:')

In [20]:
### Check if credentials are stored
'GBIF_EMAIL' in os.environ

True

### Step 1B: Create a project folder and data directories
The code below will help you get started with making a folder for your data to live in:

In [17]:
### create data directory
data_dir = os.path.join(

    ### point to your home directory
    pathlib.Path.home(),

    ### nest the data in the folder where your assignments live
    'Earth_Analytics',
    'data',
    'gbif_data')

### make the directory
os.makedirs(data_dir, exist_ok = True)

### define directory for the gbif data
gbif_dir = os.path.join(data_dir, 'gbif')

### make the directory
os.makedirs(gbif_dir, exist_ok = True)

### STEP 1C: Get the taxon key from GBIF

One of the tricky parts about getting occurrence data from GBIF is that species often have multiple names in different contexts. Luckily, GBIF also provides a Name Backbone service that will translate scientific and colloquial names into unique identifiers. GBIF calls these identifiers **taxon keys**:

1. Put the species name into the correct location in the code below.
2. Examine the object you get back from the species query. What part of it do you think might be the taxon key
3. Extract and save the taxon key.

In [23]:
### define the species name you will be using
backbone = species.name_backbone(scientificName = "Danaus plexippus")

### check it out
backbone

{'usage': {'key': '5133088',
  'name': 'Danaus plexippus (Linnaeus, 1758)',
  'canonicalName': 'Danaus plexippus',
  'authorship': '(Linnaeus, 1758)',
  'rank': 'SPECIES',
  'code': 'ZOOLOGICAL',
  'status': 'ACCEPTED',
  'genericName': 'Danaus',
  'specificEpithet': 'plexippus',
  'type': 'SCIENTIFIC',
  'formattedName': '<i>Danaus</i> <i>plexippus</i> (Linnaeus, 1758)'},
 'classification': [{'key': '1', 'name': 'Animalia', 'rank': 'KINGDOM'},
  {'key': '54', 'name': 'Arthropoda', 'rank': 'PHYLUM'},
  {'key': '216', 'name': 'Insecta', 'rank': 'CLASS'},
  {'key': '797', 'name': 'Lepidoptera', 'rank': 'ORDER'},
  {'key': '7017', 'name': 'Nymphalidae', 'rank': 'FAMILY'},
  {'key': '5133087', 'name': 'Danaus', 'rank': 'GENUS'},
  {'key': '5133088', 'name': 'Danaus plexippus', 'rank': 'SPECIES'}],
 'diagnostics': {'matchType': 'EXACT',
  'confidence': 99,
  'timeTaken': 25,
  'timings': {'nameNRank': 0,
   'sciNameMatch': 25,
   'nameParse': 9,
   'luceneMatch': 16}},
 'additionalStatus': 

Depending on the output from "backbone," you may need to modify the column in the code below to use 'usageKey' or another column:

In [ ]:
### pull out the species key
species_key = '5133088'

# ### alternate way to pull out species key
# species_key = backbone['usage']['key']

species_key

'5133088'

### Step 1D: Download data from GBIF

Downloading GBIF data is a multi-step process. However, we’ve provided you with a chunk of code that handles the API communications and caches the download. You’ll still need to customize your search.

Submit a request to GBIF using the following steps:
1. Replace `csv_file_pattern` with a string that will match **any** `.csv` file when used in the `.rglob()` method. HINT: The character `*` represents any number of any values except the file separator (e.g. `/` on UNIX systems).
2. Add parameters to the GBIF download function, `occ.download()` to limit your query to:
    - observations of 
    - from 
    - with spatial coordinates.
3. Then, run the download. This can take a few minutes. You can check your downloads by logging on to the [GBIF website](https://www.gbif.org/user/download").

In [26]:
### set file name for download
gbif_pattern = os.path.join(gbif_dir, '*.csv')

### set the code to only download your data the first time you run this chunk (since it can be slow)
if not glob(gbif_pattern):

    ### only submit one request
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:

        ### submit a request to GBIF
        gbif_query = occ.download([

            ### add your species key
            f"speciesKey = {species_key}",

            ### filter out results without geographic coordinates
            'hasCoordinate = True',

            ### choose a year to include
            f'year = 2020',
        ])

        ### grab the first result
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]

    ### wait for the download to build
    download_key = os.environ['GBIF_DOWNLOAD_KEY']

    ### use the occurrence command module in pygbif to get the metadata
    wait = occ.download_meta(download_key)['status']

    ### check if the status of the download = "SUCCEEDED"
    while not wait=='SUCCEEDED':
        wait = occ.download_meta(download_key)['status']

        ### wait before re-querying the API
        time.sleep(5)

    ### download GBIF data when it's ready
    download_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'], 
        path = gbif_dir)

    ### unzip GBIF data using the zipfile package
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path = gbif_dir)

### find the extracted .csv file path (take the first result)
gbif_path = glob(gbif_pattern)[0]

INFO:Your download key is 0035952-260928105237408
INFO:Download file size: 2237694 bytes
INFO:On disk at /Users/clairewadler/Earth_Analytics/data/gbif_data/gbif/0035952-260928105237408.zip


You might notice that the GBIF data filename isn’t very **descriptive**. At this point, you may want to clean up your data directory so that you know what the file is later on!

1. Replace ‘your-gbif-filename’ with a descriptive name.
2. Run the cell.
3. Check your data folder. Is it organized the way you want?

Tip: you can use the `shutil` package to programmatically rename the file.

In [29]:
### give the download a descriptive name
new_gbif_path = pathlib.Path(gbif_dir) / 'monarch_2020_observations'

### move file to descriptive path
shutil.move(gbif_path, new_gbif_path)

PosixPath('/Users/clairewadler/Earth_Analytics/data/gbif_data/gbif/monarch_2020_observations')

### Step 1E: Load the GBIF data into Python

Just like you did when wrangling your data from the data subset, you’ll need to load your GBIF data and convert it to a GeoDataFrame.

In [ ]:
### load the GBIF data
gbif_monarch_df = pd.read_csv(
    new_gbif_path,

    ### define value delimiter
    delimiter = '\t',

    ### set unique identifier as index
    index_col = 'gbifID',

    ### subset columns
    usecols = ['gbifID', 'decimalLatitude', 'decimalLongitude', 'month']
)

### check out the data
gbif_monarch_df.head()

,decimalLatitude,decimalLongitude,month
gbifID,,,
6579512364,30.385567,-97.706816,4.0
6579426636,40.835472,-74.176462,10.0
6579047303,43.847922,-78.965195,8.0
6578900546,38.934067,-74.954988,8.0
6578895374,28.067467,-80.736435,2.0


In [ ]:
### convert the dataframe from previous step to a geodataframe
gbif_monarch_gdf = (
    gpd.GeoDataFrame(
        gbif_monarch_df, 

        ### set geometry for geodataframe from lat/lon of GBIF record
        geometry = gpd.points_from_xy(
            gbif_monarch_df.decimalLongitude, 
            gbif_monarch_df.decimalLatitude), 

        ### set the coordinate reference system
        crs = "EPSG:4326")
    
    ### select the desired columns
    [['month', 'geometry']]
)

### check results
gbif_monarch_gdf.total_bounds

,month,geometry
gbifID,,
6579512364,4.0,POINT (-97.70682 30.38557)
6579426636,10.0,POINT (-74.17646 40.83547)
6579047303,8.0,POINT (-78.9652 43.84792)
6578900546,8.0,POINT (-74.95499 38.93407)
6578895374,2.0,POINT (-80.73644 28.06747)
...,...,...
2543082201,1.0,POINT (-81.89183 26.41157)
2542968949,1.0,POINT (-80.13173 26.35447)
2542949381,1.0,POINT (174.71857 -36.86854)


In [41]:
### Bring in the ecoregion shapefile

### create data directory
ecoregions_dir = os.path.join(

    ### point to your home directory
    pathlib.Path.home(),

    ### nest the data in the folder where your assignments live
    'Earth_Analytics',
    'data',
    'veery_migration_assignment',
    'ecoregions',
    "ecoregions.shp"
    )

### open up the ecoregions boundaries
ecoreg_gdf = gpd.read_file(ecoregions_dir)
ecoreg_gdf

,OBJECTID,ECO_NAME,BIOME_NUM,BIOME_NAME,REALM,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE,geometry
0,1.0,Adelie Land tundra,11.0,Tundra,Antarctica,AN11,1,117,9.749780,0.038948,Half Protected,#63CFAB,#9ED7C2,#257339,CC-BY 4.0,"MULTIPOLYGON (((158.7141 -69.60657, 158.71264 ..."
1,2.0,Admiralty Islands lowland rain forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Australasia,AU01,2,135,4.800349,0.170599,Nature Could Reach Half Protected,#70A800,#38A700,#7BC141,CC-BY 4.0,"MULTIPOLYGON (((147.28819 -2.57589, 147.2715 -..."
2,3.0,Aegean and Western Turkey sclerophyllous and m...,12.0,"Mediterranean Forests, Woodlands & Scrub",Palearctic,PA12,4,785,162.523044,13.844952,Nature Imperiled,#FF7F7C,#FE0000,#EE1E23,CC-BY 4.0,"MULTIPOLYGON (((26.88659 35.32161, 26.88297 35..."
3,4.0,Afghan Mountains semi-desert,13.0,Deserts & Xeric Shrublands,Palearctic,PA13,4,807,15.084037,1.355536,Nature Imperiled,#FA774D,#CC6767,#EE1E23,CC-BY 4.0,"MULTIPOLYGON (((65.48655 34.71401, 65.52872 34..."
4,5.0,Ahklun and Kilbuck Upland Tundra,11.0,Tundra,Nearctic,NE11,1,404,22.590087,8.196573,Half Protected,#4C82B6,#9ED7C2,#257339,CC-BY 4.0,"MULTIPOLYGON (((-160.26404 58.64097, -160.2673..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
842,848.0,Sulawesi lowland rain forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Australasia,AU01,2,156,150.744361,9.422097,Nature Could Reach Half Protected,#70A800,#38A700,#7BC141,CC-BY 4.0,"MULTIPOLYGON (((117.33111 -7.53306, 117.30525 ..."
843,212.0,East African montane forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Afrotropic,AF01,3,8,157.848926,5.010930,Nature Could Recover,#13ED00,#38A700,#F9A91B,CC-BY 4.0,"MULTIPOLYGON (((36.7375 -3.13, 36.7375 -3.1316..."
844,224.0,Eastern Arc forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Afrotropic,AF01,3,9,34.153333,0.890325,Nature Could Recover,#267400,#38A700,#F9A91B,CC-BY 4.0,"MULTIPOLYGON (((36.38 -8.96583, 36.38 -8.96667..."
845,79.0,Borneo montane rain forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Indomalayan,IN01,2,220,38.280990,9.358407,Nature Could Reach Half Protected,#23DB01,#38A700,#7BC141,CC-BY 4.0,"MULTIPOLYGON (((112.82394 -0.5066, 112.82298 -..."


## Wrap up Portfolio Step 1

Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

In [ ]:
%store var1 var2

Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!